In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField, StringType, IntegerType
spark=SparkSession.builder.appName("SparkJoinDataFrame").getOrCreate()

data=[('01',"Santanu",1,1000),
      ('02',"Synthia",2,1100),
      ('03',"Suman",1,900),
      ('04',"Sunny",4,800),
      ('05',"Sunil",3,1000),
      ('06',"Surya",2,1200),
      ('07',"Suresh",2,1400),
      ('08',"Suvashish",2,500)]
columns =['id','name','deptid','salary']
df=spark.createDataFrame(data,columns)
df.show()
dept_columns=['deptid','deptname']
dept_data=[(1,'Data Science'),(2,"DataEngineer"),(4,"Data Analyst")]
schema= StructType([StructField("deptid",IntegerType(),True),StructField("Deptname",StringType(),True)])
dept_df=spark.createDataFrame(dept_data,schema)
dept_df.show()


In [0]:
df.join(dept_df,df.deptid==dept_df.deptid).select(df["ID"].alias("Employee ID"),df["name"].alias("Employee Name"),dept_df["Deptname"].alias("Department")).show()

In [0]:
df.join(dept_df,df.deptid==dept_df.deptid,how='left').show()

In [0]:
from pyspark.sql.functions import coalesce,lit

df.join(dept_df, df["deptid"] == dept_df["deptid"], how="left") \
  .select(
      df["id"].alias("Employee ID"),
      df["name"].alias("Employee Name"),
      coalesce(dept_df["deptname"],lit("NonDepartment")).alias("Department Name")
  ) \
  .show()

In [0]:
df.join(dept_df, df["deptid"] == dept_df["deptid"], how="leftanti") \
\
  .show()

In [0]:
df_emp=df.join(dept_df, df["deptid"] == dept_df["deptid"], how="left") \
  .select(
      df["id"].alias("Employee ID"),
      df["name"].alias("Employee Name"),
      coalesce(dept_df["deptname"],lit("NonDepartment")).alias("Department Name"),
      df["salary"].alias("Emp salary")
  )
df_emp.show()

In [0]:
df_emp_grouped=df_emp.groupBy("Department Name").sum("Emp salary").alias("SumOfSalary")
df_emp_grouped.show()

In [0]:
from pyspark.sql import functions as F
df_emp_grade = df_emp.select(
    F.col("Employee ID").alias("Employee ID"),
    F.col("Employee Name").alias("Employee Name"),
    F.col("Department Name").alias("Department Name"),
    F.col("Emp salary").alias("Emp salary"),
    F.when(
        F.col("Emp salary") < 600, "Low Salary"
    ).when(
        (F.col("Emp salary") >= 600) & (F.col("Emp salary") < 900), "Medium Salary"
    ).when(
        (F.col("Emp salary") >= 900) & (F.col("Emp salary") < 1200), "High Salary"
    ).otherwise("Very High Salary").alias("Employee Salary Grade")
)
df_emp_grade.show()

In [0]:
df_emp_grade.groupBy("Department Name").pivot("Employee Salary Grade").avg("Emp salary").show()

User Defined Functions:-

In [0]:
df.printSchema()

In [0]:
#old implementation:-
from pyspark.sql.functions import pandas_udf, PandasUDFType
@pandas_udf("id long, salary long", PandasUDFType.GROUPED_MAP)  # doctest: +SKIP
def normalize(pdf):
    salary = pdf.salary
    return pdf.assign(salary=(salary - salary.mean()) / salary.std())
df.groupby("id").apply(normalize).show()